In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("food_sales_dataset.csv")
print(f"  Shape : {df.shape}")

  Shape : (419040, 25)


In [3]:
print(f"  Nulls      : {df.isnull().sum().sum()}")
print(f"  Duplicates : {df.duplicated().sum()}")

  Nulls      : 0
  Duplicates : 0


In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"]      = pd.to_datetime(df["date"])
 
for col in ["product_name", "category", "location",
            "customer_loyalty_tier", "product_id"]:
    df[col] = df[col].astype("category")

In [5]:
df["day_of_week"] = (df["timestamp"].dt.dayofweek + 1) % 7
df["day_name"]    = df["day_of_week"].map({
    0:"Sunday", 1:"Monday", 2:"Tuesday", 3:"Wednesday",
    4:"Thursday", 5:"Friday", 6:"Saturday"
})

In [6]:
df["season"] = df["month"].map({
    12:1, 1:1, 2:1,
     3:2, 4:2, 5:2,
     6:3, 7:3, 8:3,
     9:4,10:4,11:4
})
df["season_name"] = df["season"].map({
    1:"Winter", 2:"Spring", 3:"Summer", 4:"Fall"
})

In [7]:
df["discounted_price"] = (
    df["total_price"] * (1 - df["discount_applied"])
).round(2)

In [8]:
def time_bucket(h):
    if   0  <= h <  6: 
        return "Night"
    elif 6  <= h < 12: 
        return "Morning"
    elif 12 <= h < 17: 
        return "Afternoon"
    elif 17 <= h < 21: 
        return "Evening"
    else:              
        return "Late Night"
 
df["time_of_day"] = df["hour"].apply(time_bucket)
df["is_weekend"]  = df["day_of_week"].isin([0, 6]).astype(int)
df["year_month"]  = df["timestamp"].dt.to_period("M").astype(str)

In [9]:
assert df.isnull().sum().sum() == 0,        "Nulls found!"
assert df.duplicated().sum() == 0,          "Duplicates found!"
assert (df["quantity"] > 0).all(),          "Non-positive quantity!"
assert (df["total_price"] > 0).all(),       "Non-positive price!"
assert df["season"].isin([1,2,3,4]).all(),  "Invalid season!"
assert df["day_of_week"].between(0,6).all(),"Invalid day_of_week!"


In [10]:
df.to_csv("Cleaned_data.csv", index=False)